# Eksperimentere med Neo4j
Sjekke at vi kan kjøre Neo4J

Starte med
```
sudo apt install podman
```
Her kjører vi på egne maskiner heller enn å benytte PITs test-server.  Årsaken er enkel: Fungerer uten VDI.

## Installere Neo4J

Her kommer koden for å starte Neo4J.  Kan kjøres her, men jeg liker å ha det i en egen terminal for å kunne manipulere utenfor *notebook*.

In [ ]:
%%bash
PWD=$(pwd)
mkdir -p neo4j
mkdir -p neo4j/data
mkdir -p neo4j/logspo
mkdir -p neo4j/plugins
mkdir -p neo4j/import

# Dette laster ned to plugins fra Neo4j (som må importeres for hånd om vi skal kjøre på VDI)
# Legg merke til at vi laster to "pulgin" (APOC og GDS).
podman run \
    -p 7474:7474 -p 7687:7687 \
    --userns=keep-id \
    -e NEO4J_PLUGINS='["apoc", "graph-data-science"]' \
    -e NEO4J_dbms_security_procedures_unrestricted='gds.*,apoc.*' \
    -e NEO4J_dbms_security_procedures_allowlist='gds.*,apoc.*' \
    -e NEO4J_apoc_import_file_enabled=true \
    -v $PWD/neo4j/data:/data:Z \
    -v $PWD/neo4j/logs:/logs:Z \
    -v $PWD/neo4j/import:/import:Z \
    -v $PWD/neo4j/plugins:/plugins:Z \
    -e NEO4J_AUTH=neo4j/password \
    -d docker.io/library/neo4j:latest

: 

Når man er ferdig er det bare å kopiere identifikatoren over inn i neste kall

In [163]:
%%bash
podman kill d1349c6d65b5e1ac85ac00826fbe43f4e4e51a6553dd897b456c304a2d69734d

d1349c6d65b5e1ac85ac00826fbe43f4e4e51a6553dd897b456c304a2d69734d


Gi databasen litt tid til å starte (og laste ned utvidelsene).

Grafen er nå tilgjengelig på 
```
http://localhost:7474/browser/
```


Vi skal stort sett programmere mot Neo4j og bare unntaksvis bruke nettleseren til å "se" på grafer.

## (Kort) introduksjon til Cypher

Cypher er "SQL for grafer".  Det er strukturert på samme måte, ved at data (muligens etter transformasjoner) "flyter" gjennom programmet.  Neo4j har støtte for alt man ønsker seg, så som transaksjoner, men vi skal bare skrape tilstrekkelig på overflaten til å kunne arbeide videre på egen hånd.


### Cypher
Åpne nettleseren på adressen `http://localhost:7474/browser/`.  Dette er den interaktive måten å arbeide med en graf.

#### Noder

La oss lage en node; skriv:
```
    CREATE (n:Person {navn: "TaSK", alder: 42})
    RETURN n
```
Du får en node, og noden returneres (du ser den).  Klikk på den og se informasjonen du la inn.  Legg merke til at ute til venstre kan du velge mellom å "se" på noden, eller å få det som data, eller som "tabell".  Vi skal se nærmere på dette når vi kaller på databasen.

Lag en ny node av en annen type:
```
    CREATE (n:Firma {navn: "PIT", sektor: "Offentlig"}) 
    RETURN n
```
La oss søke etter alle noder i databasen
```
    MATCH (n) 
    RETURN n
```
Søk etter en node av en valgt type:
```
    MATCH (n:Person)
    RETURN n
```
Finne to noder av forskjellig type:
```
    MATCH (n1:Person), (n2:Firma) 
    RETURN n1, n2
```
Det reserverte ordet `MATCH` tilsvarer på mange måter `SELECT`.

#### Kanter (relasjoner)
La oss knytte de to nodene våre sammen:
```
    MATCH (n1:Person)
    MATCH (n2:Firma)
    MERGE (n1) - [r:Jobber {ansatt: 2019}] -> (n2)
    RETURN n1, n2, r
```
`MERGEP` oppretter relasjonen dersom den ikke finnes; alternativet er `CREATE`.
Klikk på "table" ute til venstre og se detaljene.

Alle relasjoner i Neo4j har retning; legg merke til (`->`).  Når vi søker kan vi benytte dette, eller betrakte relasjoner som om de var uten retning.  Detaljer senere.

Om vi hadde hatt mange noder av hver type ville vi fått mange relasjoner; det gjelder å kunne skille noder fra hverandre!

### Sletting
Neo4j er nøye på integriteten.  En node kan ikke slettes om det er relasjoner til den.  Derfor må vi slette relasjonene før vi kan slette noden.

Om vi skal slette noden `(:Person {navn: "TaSK"})` må vi først finne alle relasjoner den har:
```
    // Først
    MATCH (n:Person {navn: "TaSK"}) - [r] - ()
    DELETE r
```
Den anonyme noden (`()`) matcher enhver node, og `r` blir da alle relasjoner.

Så kan vi slette noden
```
    // Deretter
    MATCH (n:Person {navn: "TaSK"})
    DELETE n
```   

Det finnes naturligvis en innebygget snarvei for dette som vi gjør hele tiden
```
    MATCH (n:Person {navn: "TaSK"})
    DETACH DELETE n
```

### Gjøre det fra Python

Vi starter med å opprette kontakt med databasen.  Port 7687 er BOLT, en binær protokoll som biblioteket pakker ut for oss.

In [1]:
# Opprette forbindelse til databasen
from neo4j import GraphDatabase

URI = "bolt://localhost:7687"
AUTH = ("neo4j", "password")

driver = GraphDatabase.driver(URI, auth=AUTH)
driver.verify_connectivity()
# Ta et null-kall for å sjekke at det er liv - detaljer nedenfor
records, summary, keys = driver.execute_query(
    """RETURN null""")
print(f"Server: {summary.server.address}")
print("OK")

Server: 127.0.0.1:7687
OK


#### Kall til databasen

Når vi kaller (biblioteket som kaller) databasen får vi tre elementer tilbake (egentlig ett objekt med tre elementer, som her pakkes opp).  Kallet er slik:
```
records, summary, keys = driver.execute_query("CYPHER-kode")
```

`records` er det vi returnerer i Cypher.  For eksempel 
```
MATCH (p:Person) 
RETURN p.navn AS navn, p.uid AS ID
``` 
så vil vi finne `navn` og `ID` i `records`;

`summary` er meta-informasjon om spørringen, slikt som `summary.counters.nodes_created` og `summary.server.address`, og

`keys` er rett og slett navnene vi har gitt returnverdiene.  Det vil si at om `RETURN p.navn, p.ID` så vil `keys` være `['p.navn', 'p.ID']`.  Tenk på disse som navn på kolonnene dersom vi ser på `records` som data (kolonner).

Opprette tre noder og en relasjon mellom to av dem:

In [2]:
records, summary, keys = driver.execute_query(
    """
    CREATE 
        (n1:Person {navn: 'TaSK', alder: 42}),
        (n2:Person {navn: 'PålV', alder: 53}),
        (n3:Firma {navn: 'PIT', sektor: 'Offentlig'}),
        (n1) - [r:jobber {ansatt: 2019}]-> (n3)
        RETURN n1 as person, n3 as jobb, r as relasjon
    """)
#
for record in records:
    record_dict = record.data()
#
for r in record_dict:
    print(f"\t{r}: {record_dict[r]}")
#
print("Keys:")
for k in keys:
    print(f"\t{k}")
#
print("Grafen")
print(f"\tNye noder: {summary.counters.nodes_created}")
print(f"\tNye kanter: {summary.counters.relationships_created}")
      
print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")
print(f"\tÅ konsumere: {summary.result_consumed_after}ms")

	person: {'navn': 'TaSK', 'alder': 42}
	jobb: {'sektor': 'Offentlig', 'navn': 'PIT'}
	relasjon: ({'navn': 'TaSK', 'alder': 42}, 'jobber', {'sektor': 'Offentlig', 'navn': 'PIT'})
Keys:
	person
	jobb
	relasjon
Grafen
	Nye noder: 3
	Nye kanter: 1
Ressursbruk
	Kjøringen: 344ms
	Å konsumere: 58ms


Legg merke til at relasjonen (linje 3) inneholder detaljer om startnoden og sluttnoden.  Man er nesten aldri opptatt av relasjoner uavhengig av nodene relasjonen er mellom.

In [3]:
# Søke etter en node av type person som har en 
# Jobber-relasjon til en annen node med egenskapen "Offentlig".
records, summary, keys = driver.execute_query(
    """
    MATCH (n1:Person) -[r:jobber]-> (n2)
    WHERE r.ansatt > 2010
    AND n2.sektor = "Offentlig"
    RETURN n2
    """)
#
record_dict = None
for record in records:
    record_dict = record.data()
#
if not record_dict:
    print("Ingen data")
else:
    print("Returverdi")
    for r in record_dict:
        print(f"\t{r}: {record_dict[r]}")
    #
#
print("Keys:")
for k in keys:
    print(f"\t{k}")
#
print("Grafen")
print(f"\tNye noder: {summary.counters.nodes_created}")
print(f"\tNye kanter: {summary.counters.relationships_created}")
      
print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")
print(f"\tÅ konsumere: {summary.result_consumed_after}ms")

Returverdi
	n2: {'sektor': 'Offentlig', 'navn': 'PIT'}
Keys:
	n2
Grafen
	Nye noder: 0
	Nye kanter: 0
Ressursbruk
	Kjøringen: 375ms
	Å konsumere: 8ms


Denne siste kan man like godt kjøre i nettleseren.  Da får man én node.  Ved å klikke på "grafen" under noden får man de "omkringliggende" nodene (som i dette tilfelle er kun én).

In [4]:
# Slette det vi har laget og være klar til neste steg
records, summary, keys  = driver.execute_query(
    """
    MATCH (n) DETACH DELETE n
    RETURN COUNT(n)
    """)
#
for record in records:
    record_dict = record.data()
#
print("Resultat:")
for r in record_dict:
    print(f"\t{r}: {record_dict[r]}")
#
print("Keys:")
for k in keys:
    print(f"\t{k}")
#

Resultat:
	COUNT(n): 57197
Keys:
	COUNT(n)


## APOC

På samme måte som SQL er Cypher rettet mot operasjonen på objekter i en database,  Men livet er så mye mer.  *Awesome Procedures on Cypher* er et  bibliotek med hundrevis av rutiner for å gjøre "alt det andre".  Det ble lastet ned da vi startet databasen.  La oss sjekke at det hos oss.

In [7]:
records, summary, keys = driver.execute_query(
    """RETURN apoc.version() AS version;""")
print("Keys:")
for k in range(len(keys)):
    print(f"\t{keys[k]}: {records[k]}")
#

Keys:
	version: <Record version='2025.11.2'>


## Fra Networkx til Neo4j

Da skal vi bruke APOC til å lese inn en stor graf, og se litt på den.

### Eksportere fra Networkx

In [5]:
import gzip
import networkx as nx

with gzip.open("data/email.edgelist.txt.gz", "rt") as fd:
    G = nx.read_edgelist(fd, create_using=nx.DiGraph())
#
G.remove_edges_from(nx.selfloop_edges(G))
print(f"Noder: {G.number_of_nodes()}")
print(f"Kanter: {G.number_of_edges()}")

# Bruk Neo4j-syntax for å sette egenskapen
nx.set_node_attributes(G, ":Person", name="labels")
nx.set_edge_attributes(G, ":EPOST", name="label")

for n, d in G.nodes(data=True):
    # n er str
    G.nodes[n]["Navn"] = "Bruker " + n
#

# Skriv ut grafen
nx.write_graphml(G, "neo4j/import/large_graph.graphml", named_key_ids=True)
print("ok")

Noder: 57194
Kanter: 103083
ok


### Lese inn i Neo4j

In [6]:
records, summary, keys = driver.execute_query(
    """CALL apoc.import.graphml("large_graph.graphml", {storeNodeIds: true, readLabels: true})""")
# for enkelt å pakke opp svaret
for record in records:
    record_dict = record.data()
#
for k in record_dict:
    print(f"\t{k}: {record_dict[k]}")
#
print("Grafen")
print(f"\tNye noder: {summary.counters.nodes_created}")
print(f"\tNye kanter: {summary.counters.relationships_created}")
print("Begge er 0 fordi dette bare gir mening når Cypher koden Per Se genererer noder")
      
print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")
print(f"\tÅ konsumere: {summary.result_consumed_after}ms")

	file: large_graph.graphml
	source: file
	format: graphml
	nodes: 57194
	relationships: 103083
	properties: 57194
	time: 2682
	rows: 0
	batchSize: -1
	batches: 0
	done: True
	data: None
Grafen
	Nye noder: 0
	Nye kanter: 0
Begge er 0 fordi dette bare gir mening når Cypher koden Per Se genererer noder
Ressursbruk
	Kjøringen: 78ms
	Å konsumere: 2701ms


For å se resultatet, gå til browseren:
```
    MATCH(n)
    RETURN n
    LIMIT 10
```
Du skal få ti (tilfeldige) noder, og evenetuelle relasjoner mellom dem.

## Ting og tang

### Sette en index

Vi kan sette en index på én (eller flere) egenskap(er) på noder av en spesiell type.  Strengt tatt: Noder som *match*er et mønster.

In [7]:
records, summary, keys = driver.execute_query(
    """
    CREATE INDEX
    IF NOT EXISTS 
    FOR (n:Person) ON (n.Navn);
    """)      
print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")

Ressursbruk
	Kjøringen: 51ms


Minner om at en index endrer kjøretiden fra $O(N)$ til $O(log~n)$.

### Hente data
Sjekke noen velkjente noder:
- Bruker 23
- Bruker 26
- Bruker 15
- Bruker 6

Den "andre" måten å bruke driveren på, er gjennom sesjoner.  Resultatet må hentes i konteksten

Her henter vi fire (velkjente) noder, og hvor mange relasjoner hver node har.

Legg merke til hvordan `p` er bundet til én node om gangen, slik at `(p)--()` (som er settet av relasjoner `p` har) gir mening.

In [9]:
with driver.session(database="neo4j") as session:
    resultat = session.run("""
    MATCH (p:Person)
    WHERE p.Navn IN ["Bruker 23", "Bruker 26", "Bruker 15", "Bruker 6"]
    RETURN p.Navn AS navn, COUNT {(p)--()} as naboer
    """)
    det_hele = []
    for r in resultat:
        det_hele += [r]
    #
    for s in resultat.consume().gql_status_objects:
        print(f"Hvordan gikk det: {s}")
    #
#
# Utenfor "scope"
for d in det_hele:
    print(d)
#

Hvordan gikk det: note: successful completion
<Record navn='Bruker 23' naboer=137>
<Record navn='Bruker 26' naboer=99>
<Record navn='Bruker 15' naboer=75>
<Record navn='Bruker 6' naboer=263>


Gå i nettleseren:
```
MATCH (p:Person)
WHERE p.Navn = "Bruker 6"
RETURN p
```
Klikk på noden for å se 100 av de 263 relasjonene.

La oss ha antall kanter lett tilgjengelig.  Først regner vi ut for hver node hvor mange relasjoner den har, lagrer det på noden, og til slutt setter på en index.

In [10]:
# Legg inn
records, summary, keys = driver.execute_query(
    """
    MATCH (n:Person)
    SET n.antallKanter = COUNT { (n)--() }
    """)
for r in records:
    # Skal være tom
    innhold = r.data()
    print(innhold)
#
records, summary, keys = driver.execute_query(
    """
    CREATE INDEX
    IF NOT EXISTS 
    FOR (n:Person) ON (n.antallKanter);
    """)
for r in records:
    # Skal være tom
    innhold = r.data()
    print(innhold)
#
print("ok")

ok


Det er ikke bare noder som kan returneres; essensen i grafer (i motsetning til tabller) er at relasjonene er eksplisitte.

Først søker vi etter et mønster hvor en node (`()`) som har en relasjon til noden "Bruker 6", og returnerer dem (altså relasjonene).

Dernest etter noder (`(p)`) som har en relasjon til samme, og returner nodene.

In [11]:
records, summary, keys = driver.execute_query(
    """
    MATCH 
    p=()-->(:Person {Navn:"Bruker 6"}) 
    RETURN p;
""")
# Returnerer et sett av STIER som ender i noden identifisert med id=6
for r in records:
    innhold = r.data()
    print(innhold)
    break # Første er nok
#
records, summary, keys = driver.execute_query(
    """
    MATCH (p)-->(:Person {Navn:"Bruker 6"}) 
    RETURN p;
""")
# Returnerer et sett av NODER som har relasjoner til noden identifisert med id=6
for r in records:
    innhold = r.data()
    print(innhold)
    break # Første er nok
#
print("ok")

{'p': [{'id': '40276', 'antallKanter': 2, 'Navn': 'Bruker 40276'}, 'EPOST', {'id': '6', 'antallKanter': 263, 'Navn': 'Bruker 6'}]}
{'p': {'id': '40276', 'antallKanter': 2, 'Navn': 'Bruker 40276'}}
ok


Legg merke til at relasjonen (første linje) inkluderer start-noden, relasjonen, og så ende-noden.

## GDS

Cypher egner seg til "enkle ting".  Det vil si transaksjoner på og med noder, men er ikke egnet til å implementere algoritmer.  **Graph Data Science** er implementert som et bibliotek, og kjører inne i Neo4J og har derfor fri tilgang til alle interne datastrukturer.

In [13]:
# Sjekke at GDS har blitt lastet ned riktig
records, summary, keys = driver.execute_query(
    """
    CALL gds.version();
    """)

print("Keys:")
for k in range(len(keys)):
    print(f"\t{keys[k]}: {records[k]}")
#

Keys:
	gdsVersion: <Record gdsVersion='2.24.0'>


In [27]:
# For ordens skyld, i tilfelle databasen har blitt brukt, la oss slette
# alle GDS-grafer.  
# Detaljer om litt
records, summary, keys = driver.execute_query(
    """
    CALL gds.graph.list() YIELD graphName
    WITH graphName
    CALL gds.graph.drop(graphName) YIELD graphName AS borte
    RETURN borte
    """)
# Trolig tom
for r in records:
    innhold = r.data()
    print(innhold)
#
print("ok")


{'borte': 'ViktigGraf'}
ok



Fordi mange graf-algoritmer i praksis bruker alle noder i grafen, er valget med å kjøre i en definert sub-graf og kreve at den er i hukommelsen, et design som gir mening.  Tross alt er tilgang til data i hukommelsen minst fire størrelsesordner raskere, og ett eneste søk på disk kan ødelegge alt.  For å gjøre dette mulig er flyten delt i tre, og eksplisitt funksjonalitet for testing tilgjengelig.

Dersom en sub-graf er for stor til å kunne kjøres i minnet, da må man falle tilbake til det vi er vant til når det gjelder databaser: Vente eller være kreative.

Flytens tre (fire) steg er:
- Konstruere (sub)grafen i hukommelsen ved å velge noder og relasjoner som er relevante.  Det gjøres enten med primitiver tilgjengelig i GDS dersom subgrafen skal bestå av  "enkle ting".  Eller brukes Cyhper til å velge ut noder og relasjoner;
- For sikkerhets skyld bør man be om et estimat på algoritmen som skal kjøres.  Estimatet gjøres ved å estimere algoritmen opp mot subgrafen som er laget;
- Kjøre algoritmen, og
- Fjerne subgrafen når det ikke lenger er brhov for den.

Algoritmen bør ha noen sideeffekter.  Det er fire måter å skape dem:
- **stream**: Data returneres til kalleren, som enten er Python-koden eller i nettleseren;
- **stats**: Returnerer statestikk (metainformasjon) heller enn noder og relasjoner;
- **mutate**: Skriver endringer tilbake til subgrafen i hukommelsen, og
- **write**: Skriver endringer tilbake i selve databasen.

En sub-graf i minnet kaller vi en *projeksjon* (fordi det er hva det er).

### Eksempel på lasting med merkede noder

Det enkleste er å merke det man vil ha med, og så laste dem inn direkte.  Vi har allerede satt egenskapen `antallKanter` på hver node, (og satt på en index).  Vi bruker `antallKanter` til å merke noder som har mange relasjoner.

#### Merke nodene

In [31]:
records, summary, keys = driver.execute_query(
    """
    MATCH (p:Person) // Kun personer, takk
    WHERE p.antallKanter > 100
    SET p:Viktig
    RETURN COUNT(p)
    """)
for s in resultat.consume().gql_status_objects:
    print(f"Hvordan gikk det: {s}")
#
for r in records:
    print(r.data())
#
print("ok")

Hvordan gikk det: note: successful completion
{'COUNT(p)': 209}
ok


#### Lage subgrafen
Så laster vi de nodene sammen med (kun) `EPOST`-relasjonene for det kan jo være andre relasjoner i databasen:

In [32]:
records, summary, keys = driver.execute_query(
    """
    CALL gds.graph.project(
        'ViktigGraf',  // Gi projeksjonen et navn
        ['Viktig'],    // Første "søk": Noder
        ['EPOST']      // Andre  "søk": Kanter
    )
    """)
for s in resultat.consume().gql_status_objects:
    print(f"Hvordan gikk det: {s}")
#
print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")
print("ok") 


Hvordan gikk det: note: successful completion
Ressursbruk
	Kjøringen: 1ms
ok


In [33]:
# Hva er status
records, summary, keys = driver.execute_query(
    """CALL 
        gds.graph.list('ViktigGraf') 
        YIELD graphName, nodeCount, relationshipCount
        RETURN graphName, nodeCount, relationshipCount
    """
)
for r in records:
    innhold = r.data()
    print(innhold)
#
print("ok") 

{'graphName': 'ViktigGraf', 'nodeCount': 209, 'relationshipCount': 1841}
ok


#### Et estimat på kjøringen

Vi så på `pageRank`, og vi kan be om et estimat for å få det beregnet.

In [33]:
# Et estimat
records, summary, keys = driver.execute_query(
    """
    CALL gds.pageRank.stream.estimate (  // Få estimat på å sende resultatene tilbake til meg (stream)
        'ViktigGraf',
        {
        maxIterations: 20,   // Parametre for algoritmen
        dampingFactor: 0.85  // Ditto
        }
        )
    YIELD nodeCount, bytesMin, bytesMax, requiredMemory
    """)
for s in resultat.consume().gql_status_objects:
    print(f"Hvordan gikk det: {s}")
print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")
for r in records:
    innhold = r.data()
    print(innhold)
    break # Første er nok
#
print("ok") 


Hvordan gikk det: note: successful completion
Ressursbruk
	Kjøringen: 57ms
{'nodeCount': 209, 'bytesMin': 5872, 'bytesMax': 5872, 'requiredMemory': '5872 Bytes'}
ok


Tallene er neglisjerbare.

#### Hente data fra subgrafen

Her ser vi hvordan vi kaller på GDS.  Syntaxen er den samme som for APOC.  

Først kjøre *pageRank* på nodene i projeksjonen, og oppdatere dem med verdien (*mutate*).  Det reservberte ordet `mutateProperty` står i dokumentasjonen.

In [34]:
# La oss få data
records, summary, keys = driver.execute_query(
    """
    CALL gds.pageRank.mutate('ViktigGraf',  // OBS: Mutate (skrive til projeksjonen)
        {mutateProperty: 'HvorViktig'}
    )
    """)
for s in resultat.consume().gql_status_objects:
    print(f"Hvordan gikk det: {s}")
#
print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")
print("ok") 


Hvordan gikk det: note: successful completion
Ressursbruk
	Kjøringen: 2ms
ok


Så ber vi om å få verdiene strømmet (*stream*):

In [ ]:
# Hente de fem viktigste nodene
records, summary, keys = driver.execute_query(
    """
    CALL gds.graph.nodeProperty.stream(  // Stream denne gangen
        'ViktigGraf', 
        'HvorViktig')
    YIELD nodeId, propertyValue // disse verdiene står i dokumentasjonen
    RETURN gds.util.asNode(nodeId).Navn AS Navn, propertyValue AS Verdi
    ORDER BY Verdi DESC
    LIMIT 5
    """)
for s in resultat.consume().gql_status_objects:
    print(f"Hvordan gikk det: {s}")
print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")
for r in records:
    innhold = r.data()
    print(innhold)
#
print("ok") 


Hvordan gikk det: note: successful completion
Ressursbruk
	Kjøringen: 49ms
{'Navn': 'Bruker 11798', 'Verdi': 8.272769620366098}
{'Navn': 'Bruker 12586', 'Verdi': 2.593843757509939}
{'Navn': 'Bruker 14603', 'Verdi': 2.475295656143867}
{'Navn': 'Bruker 880', 'Verdi': 1.6377652947797572}
{'Navn': 'Bruker 737', 'Verdi': 1.6068969822908044}
ok


_Score_ er større enn 0 fordi summen av alle blir $N$ (opprinnelig satt til 1 på alle noder).


### Eksempel uten merkelapp

Vi kan kjøre Cypher som en del av byggingen av en sub-graf; dette er et KI-forslag.  Vi trenger to (2) Cypher-kall: Ett for noder og ett for kanter.
```
CALL gds.graph.project(
  'ViktigGraf',            // Navnet på subgrafen
  'MATCH (p:Person) 
   WHERE COUNT { (p)-[:EMAIL]-() } > 100   // Bare noder med mer enn 100 eposter
   RETURN elementId(p) AS id',             // sendes til neste steg
   // Her finner vi hvilke kanter som skal være med.
   // Denne krever to tellinger for å sikre at det ikke forsøkes å lage en relasjon til noder
   // som ikke er med (har mindre enn 100 eposter).
  'MATCH (p1:Person)-[r:EMAIL]-(p2:Person) 
   WHERE COUNT { (p1)-[:EMAIL]-() } > 100  // 
     AND COUNT { (p2)-[:EMAIL]-() } > 100
   RETURN elementId(p1) AS source, elementId(p2) AS target, type(r) AS type' // Relationship query
)
```

Om denne fremgangsmåten er raskere vil avhenge av tettheten i grafen; det er slikt erfaring vil fortelle.

### Eksempel på å skrive tilbake til databsen

Her ser vi hvordan data skrives tilbake til de nodene det gjelder.  La oss gjøre det i nettleseren for å demonstrere.

```
CALL gds.pageRank.write(
  'ViktigGraf',
  {
    writeProperty: 'HvorViktig' // Hva skal egenskapen på noden hete
  }
)
YIELD nodePropertiesWritten, ranIterations // Dette er returverdien og ikke sideeffekten!
```
Så kan vi hente resultatene med Cypher:
```
MATCH (p:Person)
WHERE p.HvorViktig IS NOT NULL
RETURN p.Navn, p.HvorViktig
ORDER BY p.HvorViktig DESC
LIMIT 10
```


#### Fjerne subgrafen

Bør kjøres for å se håndteringen av at det ikke er noe å fjerne.

In [43]:
# Fjerne subgrafen
records, summary, keys = driver.execute_query(
    """
    CALL gds.graph.drop('ViktigGraf', false)  // false = Ikke få feil om den ikke finnes
    YIELD graphName
    """)
for s in resultat.consume().gql_status_objects:
    print(f"Hvordan gikk det: {s}")
#
if not records:
    print("Allerede slettet")
else:
    print(f"Navnet på subgrafen: {records}")
#
print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")

print("ok") 



Hvordan gikk det: note: successful completion
Allerede slettet
Ressursbruk
	Kjøringen: 0ms
ok


#### Fjerne merkelappene

Vi skrev tilbake til databasen, og la oss fjerne det også.

In [45]:
# Til slutt,gjerne merkelappen
records, summary, keys = driver.execute_query(
    """
    MATCH (p:Viktig) 
    REMOVE p:Viktig
    RETURN COUNT (p);
    """)
for s in resultat.consume().gql_status_objects:
    print(f"Hvordan gikk det: {s}")
#
for rec in records:
    for r in rec:
        print(f"Slettet: {r}")
#
print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")

print("ok") 

Hvordan gikk det: note: successful completion
Slettet: 0
Ressursbruk
	Kjøringen: 76ms
ok


### GDS til å lage noe som ligner på nettleseren
Et større og mer komplisert eksempel.
Dette er hvordan Neo4J tegner grafen i en *browser*:
![Grafen](data/Neo4j-graf.png)

Det er åpenbart en sub-graf.  Vi skal forsøke å gjenskape dette bildet.

1. Lage en subgraf bestående av alle personer (nå er det ikke noe annet i denne grafen);
2. Finne klikker or merke nodene med hvilken klikk de er med i;
3. Telle opp hvor mange medlemmer det er i hver klikk, og legge det inn i hver node, og
4. Fjerne merkingen på klikker som er "små".

Når vi (i nettleseren) ber om å få se klikkene kommer bare de store (og "støyen" filtreres ut).

#### Lag subgrafen

Hente ut Personer og kanter.  Vi er ikke interessert i retningen på kanten.

In [ ]:
records, summary, keys = driver.execute_query(
    """
    // Subgrafen heter EpostNettverk
    CALL gds.graph.project(
        'EpostNettverk', // Navnet på subgrafen
        'Person',  // Første søk: Nodene
        {
            EPOST: {
                orientation: 'UNDIRECTED' // Andre søk: relasjonene.  Konverteres til uten retning
            }
        }
    )
    YIELD graphName, nodeCount, relationshipCount;
    """)
for s in resultat.consume().gql_status_objects:
    print(f"Hvordan gikk det: {s}")
#
# for enkelt å pakke opp svaret
for record in records:
    record_dict = record.data()
#
for k in record_dict:
    print(f"\t{k}: {record_dict[k]}")
#

print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")

print("ok") 

Hvordan gikk det: note: successful completion
	graphName: EpostNettverk
	nodeCount: 57194
	relationshipCount: 206166
Ressursbruk
	Kjøringen: 0ms
ok


#### Identifisere gjengene

In [ ]:
# Burk Louvain for å finne gjenger og skrtive dem tilbake i databasen
records, summary, keys = driver.execute_query(
    """
    CALL gds.louvain.write(
        'EpostNettverk', 
        {
            writeProperty: 'GjengID'  // For hver node,skrive hvilken gjeng noden er med i
        }
    )
    YIELD communityCount, modularity;
    """)
for s in resultat.consume().gql_status_objects:
    print(f"Hvordan gikk det: {s}")
#
# for enkelt å pakke opp svaret
for record in records:
    record_dict = record.data()
#
for k in record_dict:
    print(f"\t{k}: {record_dict[k]}")
#

print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")

print("ok") 


Hvordan gikk det: note: successful completion
	communityCount: 288
	modularity: 0.7458673476585517
Ressursbruk
	Kjøringen: 4ms
ok


#### Fjerne små gjenger

In [ ]:
# Fjerne gjenger med mindre enn 10 medlemmer (tilfeldig valgt tall)
records, summary, keys = driver.execute_query(
    """
    MATCH (n)
    WITH 
        n.GjengID AS antall,   // Samler sammen alle noder med samme GjengID
        count(n) AS medlemmer  // Teller hvor mange i hver gjeng
    WHERE medlemmer < 10
    MATCH (m {GjengID: antall})
    REMOVE m.GjengID;
    """)
for s in resultat.consume().gql_status_objects:
    print(f"Hvordan gikk det: {s}")
#
# for enkelt å pakke opp svaret
for record in records:
    record_dict = record.data()
#
for k in record_dict:
    print(f"\t{k}: {record_dict[k]}")
#

print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")

print("ok") 


Hvordan gikk det: note: successful completion
	communityCount: 288
	modularity: 0.7458673476585517
Ressursbruk
	Kjøringen: 1580ms
ok


Nå er det bare å limem dette inn i nettleseren for å få nesten samme bilde som over
```
MATCH (n) WHERE n.GjengID IS NOT NULL RETURN n LIMIT 1000
```


#### Fjerne subgrafen

In [204]:
# Fjerne subgrafen
records, summary, keys = driver.execute_query(
    """
    CALL gds.graph.drop('EpostNettverk', false)  // false = Ikke få feil om den ikke finnes
    YIELD graphName
    """)
for s in resultat.consume().gql_status_objects:
    print(f"Hvordan gikk det: {s}")
#
if not records:
    print("Allerede slettet")
else:
    print(f"Navnet på subgrafen: {records}")
#
print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")

print("ok") 



Hvordan gikk det: note: successful completion
Allerede slettet
Ressursbruk
	Kjøringen: 0ms
ok
